# ZEDprofiler Walkthrough

ZEDprofiler is a CPU-first toolkit for extracting morphological features from 3D fluorescence microscopy images.
It is designed for high-content and high-throughput workflows where images are volumetric z-stacks with multiple spectral channels and segmentation labels.

This notebook walks through the full feature extraction pipeline end-to-end:

1. **Loading** — configure image sets and segmentation objects using `ImageSetLoader` and `ObjectLoader`
2. **Featurization** — extract six classes of morphological features (colocalization, granularity, intensity, neighbors, texture, volume/size/shape)
3. **Merging** — combine all feature DataFrames into a single profile table keyed by image set and object ID

Each section includes a brief description of what the feature class measures and what the key parameters control.
The notebook runs on synthetic data by default so no external data is required.

## Imports

In [ ]:
import logging
import os
import pathlib

import numpy as np
import pandas as pd
import tifffile
from itables import show

from zedprofiler.IO.loading_classes import (
    ImageSetConfig,
    ImageSetLoader,
    ObjectLoader,
    TwoObjectLoader,
)

from zedprofiler.featurization.colocalization import compute_colocalization
from zedprofiler.featurization.granularity import compute_granularity
from zedprofiler.featurization.intensity import compute_intensity
from zedprofiler.featurization.neighbors import compute_neighbors
from zedprofiler.featurization.texture import compute_texture
from zedprofiler.featurization.volumesizeshape import compute_volume_size_shape


logging.basicConfig(level=logging.INFO)

## Define Paths and Parameters

Set `GENERATE_SYNTHETIC_DATA = True` to run the notebook on randomly generated 3D arrays.
Set it to `False` and update the paths below to run on your own image sets.

The `channel_mapping` dictionary maps logical channel/compartment names to substrings found in your image filenames.
ZEDprofiler uses these keys throughout the pipeline to identify which image corresponds to which biological channel or segmentation label.

In [2]:
# use your own data directory or generate synthetic data for testing?
GENERATE_SYNTHETIC_DATA = True

In [3]:
if not GENERATE_SYNTHETIC_DATA:
    # put your own data directories here if you want to test with real data
    image_set_path = pathlib.Path(
        os.path.expanduser(
            "~/mnt/bandicoot/NF1_organoid_data/data/NF0014_T1/zstack_images/C4-2"
        )
    ).resolve()
    label_set_path = pathlib.Path(
        os.path.expanduser(
            "~/mnt/bandicoot/NF1_organoid_data/data/NF0014_T1/segmentation_masks/C4-2"
        )
    ).resolve()

    # mapping to unique keys in the image names to extract
    # what that given image is
    channel_mapping = {
        "DNA": 405,
        "ER": 488,
        "AGP": 555,
        "Mito": 640,
        "Nuclei": "nuclei_",
        "Cytoplasm": "cytoplasm_",
        "Cell": "cell_",
        "Organoid": "organoid_",
    }

    CHANNEL1 = "DNA"
    CHANNEL2 = "ER"
    COMPARTMENT = "Nuclei"
else:
    notebook_root = pathlib.Path.cwd().resolve()
    image_set_path = (notebook_root / "test_data" / "dummy_image_set").resolve()
    label_set_path = (notebook_root / "test_data" / "dummy_label_set").resolve()
    channel_mapping = {
        "Channel1": "channel1",
        "Channel2": "channel2",
        "Nuclei": "nuclei_labels",
    }
    CHANNEL1 = "Channel1"
    CHANNEL2 = "Channel2"
    COMPARTMENT = "Nuclei"
    if not image_set_path.is_dir() or not label_set_path.is_dir():
        # generate and save dummy data if real data is not available
        channel1_array = (np.random.rand(100, 100, 10) * 255).astype(np.uint8)
        channel2_array = (np.random.rand(100, 100, 10) * 255).astype(np.uint8)
        nuclei_labels = np.random.randint(0, 5, size=(100, 100, 10), dtype=np.uint16)
        image_set_path.mkdir(parents=True, exist_ok=True)
        label_set_path.mkdir(parents=True, exist_ok=True)
        tifffile.imwrite(image_set_path / "channel1.tif", channel1_array)
        tifffile.imwrite(image_set_path / "channel2.tif", channel2_array)
        tifffile.imwrite(label_set_path / "nuclei_labels.tif", nuclei_labels)

## Loading Objects and Image Sets

ZEDprofiler uses three loader classes to organize image data before featurization:

- **`ImageSetConfig`** — declares which keys are raw intensity channels (`raw_image_key_name`) and which are segmentation labels (`label_key_name`). This config is passed to the loader so it knows how to partition the channel mapping.
- **`ImageSetLoader`** — loads the full image set from disk (or from in-memory arrays) and stores each channel and label as a numpy array. The `anisotropy_spacing` parameter describes the physical voxel size in `[z, y, x]` order and is used by downstream features (e.g. neighbors, surface area) that require real-world distances.
- **`ObjectLoader`** — a lightweight view over one channel + one compartment within a loaded image set. Most featurization functions accept an `ObjectLoader`.
- **`TwoObjectLoader`** — a paired view over two channels within one compartment, required by colocalization which compares signal across two channels simultaneously.

In [4]:
isc = ImageSetConfig(
    image_set_name="test_image_set",
    raw_image_key_name=[CHANNEL1, CHANNEL2],
    label_key_name=[COMPARTMENT],
)

image_set_loader = ImageSetLoader(
    anisotropy_spacing=[1, 0.1, 0.1],
    channel_mapping=channel_mapping,
    image_set_path=image_set_path,
    label_set_path=label_set_path,
    config=isc,
    # we can also directly pass arrays.
    # This is what ZEDprofiler expects when the images are multichannel.
    # Load the image arrays in and pass them in directly instead
    # of loading from path.
    image_set_array=None,
    label_set_array=None,
)

In [5]:
ol = ObjectLoader(
    image_set_loader=image_set_loader,
    channel_name=CHANNEL1,
    compartment_name=COMPARTMENT,
)

In [6]:
coloc_loader = TwoObjectLoader(
    image_set_loader=image_set_loader,
    compartment=COMPARTMENT,
    channel1=CHANNEL1,
    channel2=CHANNEL2,
)

## Colocalization

![Colocalization](../media/3D_Colocalization_Features.png)


Colocalization measures the degree to which two fluorescent channels occupy the same spatial locations within a segmented compartment.
This is useful for quantifying co-expression or co-localization of proteins in 3D cell images.

ZEDprofiler computes several colocalization coefficients per object pair:

- **Pearson correlation** — linear correlation of pixel intensities between the two channels
- **Manders coefficients (M1, M2)** — fraction of channel 1 (or 2) signal that overlaps with channel 2 (or 1)
- **Costes thresholded Manders** — Manders coefficients computed above a statistically determined threshold
- **Overlap coefficient** — combined spatial and intensity overlap
- **Rank-weighted colocalization (RWC)** — robust to intensity differences, weights overlap by rank

Key parameters:
- `thr` — minimum pixel intensity threshold below which pixels are excluded
- `fast_costes` — `"Accurate"` uses the full bisection algorithm for the Costes threshold; `"Fast"` uses a linear approximation

In [7]:
colocalization_features = compute_colocalization(
    two_object_loader=coloc_loader,
    thr=15,
    fast_costes="Accurate",
    channel1=CHANNEL1,
    channel2=CHANNEL2,
)

## Granularity

![Granularity](../media/3D_Granularity_Features.png)


Granularity characterizes the texture of an object at multiple spatial scales by repeatedly applying morphological erosions and measuring how much signal is removed at each scale.
The result is a spectrum of values — one per scale — that captures whether an object's intensity is concentrated in fine, coarse, or mixed-scale structures.
This is analogous to CellProfiler's granularity module and is particularly useful for distinguishing organelle morphology.

Key parameters:
- `granular_spectrum_length` — number of erosion scales to compute (default 16); higher values capture coarser structure
- `radius` — radius of the structuring element used for background subtraction before the spectrum is computed
- `subsample_size` — fraction of the image used during spectrum computation (speeds up processing on large volumes)
- `image_sample_size` — fraction used for background estimation
- `mask_threshold` — threshold applied after interpolation to define the object mask

In [8]:
granularity_features = compute_granularity(
    object_loader=ol,
    radius=10,  # radius of the structuring element for background
    # removal (CellProfiler default)
    granular_spectrum_length=16,  # range of the granular spectrum
    subsample_size=0.25,  # subsample the image for faster processing
    image_sample_size=0.25,  # further subsample for background removal
    mask_threshold=0.9,  # threshold for determining mask after interpolation
    verbose=False,  # print out intermediate steps and values for debugging
)

## Intensity

![Intensity](../media/3D_Intensity_Features.png)


Intensity features summarize the distribution of pixel intensities within each segmented object.
They provide a compact description of how bright an object is, how variable its signal is, and how that signal is spatially distributed relative to the object's edges.

Features computed per object include:
- **Mean, median, standard deviation, max, min** of voxel intensities
- **Integrated intensity** — sum of all voxel intensities (correlated with object volume)
- **Mass displacement** — distance between the object's geometric centroid and its intensity-weighted centroid, indicating asymmetric signal distribution
- **Lower/upper quartile intensities** — robust measures of the intensity distribution

In [9]:
intensity_features = compute_intensity(
    object_loader=ol,
)

## Neighbors

![Neighbors](../media/3D_Neighbors_Features.png)


Neighbor features describe the spatial relationships between objects within a compartment.
For each object, ZEDprofiler identifies all other objects whose centroids fall within a given distance and computes summary statistics over that neighborhood.

Features include counts of neighbors, mean and standard deviation of distances to neighbors, and the angle between each object and its closest neighbor.

Key parameters:
- `distance_threshold` — maximum centroid-to-centroid distance (in voxels) for two objects to be considered neighbors
- `anisotropy_factor` — the ratio of z-spacing to xy-spacing (`anisotropy_spacing[0] / anisotropy_spacing[1]`); used to rescale z-distances to physical units before computing Euclidean distances, ensuring accurate neighbor relationships in anisotropic volumes

In [10]:
neighbors_features = compute_neighbors(
    object_loader=ol,
    distance_threshold=10,
    anisotropy_factor=image_set_loader.anisotropy_spacing[0]
    / image_set_loader.anisotropy_spacing[
        1
    ],  # z to xy spacing ratio for distance calculation
)

## Texture

![Texture](../media/3D_Texture_Features.png)


Texture features capture the spatial arrangement of intensities within an object using the gray-level co-occurrence matrix (GLCM).
The GLCM records how often pairs of pixel intensities occur at a given spatial offset, and summary statistics computed from it describe properties like smoothness, contrast, and periodicity.

ZEDprofiler computes GLCM-based features (e.g. angular second moment, contrast, correlation, entropy, homogeneity) at multiple offsets and directions in 3D, then aggregates across directions.

Key parameters:
- `distance` — the pixel offset at which co-occurrence is measured; larger values capture coarser texture patterns
- `grayscale` — the number of intensity bins used to quantize the image before computing the GLCM; fewer bins are faster but lose intensity resolution

In [11]:
texture_features = compute_texture(
    object_loader=ol,
    distance=3,
    grayscale=256,
)

## Volume, Size, and Shape

![Volume, Size, and Shape](../media/3D_Volume_Shape_Features.png)


Volume, size, and shape features characterize the 3D geometry of each segmented object.
They are computed using `skimage.measure.regionprops` on the label image, with surface area calculated via marching cubes.

Features per object include:
- **Volume** — total number of voxels in the object
- **Bounding box** — min/max extents in x, y, z
- **BboxVolume** — volume of the axis-aligned bounding box
- **Centroid** — geometric center in x, y, z
- **Extent** — ratio of object volume to bounding box volume (compactness)
- **Euler number** — topological measure of holes/tunnels in the object
- **Equivalent diameter** — diameter of a sphere with the same volume
- **Surface area** — estimated via marching cubes on the object boundary

In [12]:
volume_size_shape_features = compute_volume_size_shape(
    image_set_loader=image_set_loader,
    object_loader=ol,
)

## Merging All Features into a Single Profile

Each featurization function returns a dictionary that can be converted to a DataFrame.
All DataFrames share two index columns — `Metadata_Experiment_ImageSet` and `Metadata_Object_ObjectID` — which are used as merge keys.

The final merged DataFrame contains one row per object and one column per feature, following the ZEDprofiler naming convention:
`{Compartment}_{Channel}_{FeatureType}_{Measurement}`

See the [Feature Schema](../features/Feature_Naming_Convention.md) for the full specification of how columns are named and structured.

The cell below also estimates the total expected feature count when scaling to a full experiment with multiple compartments and channels, giving a sense of the profile dimensionality at scale.

In [13]:
coloc_df = pd.DataFrame(colocalization_features)
granularity_df = pd.DataFrame(granularity_features)
intensity_df = pd.DataFrame(intensity_features)
neighbors_df = pd.DataFrame(neighbors_features)
texture_df = pd.DataFrame(texture_features)
volume_size_shape_df = pd.DataFrame(volume_size_shape_features)
# subtract 2 from each df shape to account for image_set and object_id
# columns that will be used for merging
size_of_all_dfs = [
    df.shape[1] - 2
    for df in [
        coloc_df,
        granularity_df,
        intensity_df,
        neighbors_df,
        texture_df,
        volume_size_shape_df,
    ]
]
# add the 2 back to account for the image_set and object_id columns
# that will be retained in the final merged DataFrame
expected_num_columns = sum(size_of_all_dfs) + 2

In [ ]:
columns_to_merge = ["Metadata_Experiment_ImageSet", "Metadata_Object_ObjectID"]

for df in [
    coloc_df,
    granularity_df,
    intensity_df,
    neighbors_df,
    texture_df,
    volume_size_shape_df,
]:
    if not all(col in df.columns for col in columns_to_merge):
        raise ValueError(
            f"DataFrame is missing required columns for merging: {columns_to_merge}"
        )
# merge all features into a single DataFrame
all_features_df = pd.merge(
    coloc_df,
    pd.merge(
        granularity_df,
        pd.merge(
            intensity_df,
            pd.merge(
                neighbors_df,
                pd.merge(
                    texture_df, volume_size_shape_df, on=columns_to_merge, how="inner"
                ),
                on=columns_to_merge,
                how="inner",
            ),
            on=columns_to_merge,
            how="inner",
        ),
        on=columns_to_merge,
        how="inner",
    ),
    on=columns_to_merge,
    how="inner",
)
if all_features_df.shape[1] != expected_num_columns:
    raise ValueError(
        f"""
        Merged DataFrame has {all_features_df.shape[1]} columns
        but expected {expected_num_columns} based on individual DataFrames"""
    )
print(f"Shape: {all_features_df.shape[0]} objects x {all_features_df.shape[1]} features")
show(all_features_df)

In [15]:
expected_num_compartments = 4
expected_num_of_channels = 4
expected_num_of_total_features = (
    expected_num_columns - 2
)  # subtract 2 for image_set and object_id columns
expected_num_of_total_features = (
    expected_num_of_total_features
    * expected_num_compartments
    * expected_num_of_channels
    + 2
)
expected_num_of_total_features

3698

In [16]:
if all_features_df.shape[1] != expected_num_columns:
    raise ValueError(
        f"Expected {expected_num_columns} columns but got {all_features_df.shape[1]}"
    )